# 앵커 풀 분석

전수 라벨링에 쓰는 기준 앵커 풀(`anchor_pool_v2.jsonl`)의 구성과, 층화 인출이 실제로 어떤 앵커를 꺼내는지 확인합니다.

## 층화 인출 방식 (결정 14)

```
어절 1~2gram TF-IDF + 문자 3~4gram(char_wb) TF-IDF  →  코사인 유사도
  → 동일 문서 앵커 마스킹(-1.0)          (결정 10, 누수 차단)
  → 라벨별 최고 유사도 1건씩 = 총 3건     (결정 14, 다수 라벨 편향 억제)
```

두 벡터 블록이 각각 L2 정규화되므로 결합 유사도는 어절 유사도와 문자 유사도의 평균과 같습니다.

## 보는 것

1. 풀 구성 — 라벨·문서·출처 등급 분포
2. 실제 인출 결과 — 전체 1,024건에 대해 어떤 앵커가 몇 번 쓰이는가
3. 인출 유사도 분포 — 꺼내온 앵커가 실제로 유사한가

API를 호출하지 않습니다. 로컬 TF-IDF 연산만 사용합니다.

In [ ]:
from pathlib import Path
import json
import sys
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib import font_manager

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from scripts.labeling.anchor_pool import load_anchor_pool
from scripts.labeling.anchor_retriever import PureTfidfAnchorRetriever, retriever_config

installed = {f.name for f in font_manager.fontManager.ttflist}
plt.rcParams['font.family'] = next(
    f for f in ['Malgun Gothic', 'NanumGothic', 'Noto Sans CJK KR', 'AppleGothic', 'DejaVu Sans']
    if f in installed
)
plt.rcParams['axes.unicode_minus'] = False

POOL = ROOT / 'data' / 'anchors' / 'anchor_pool_v2.jsonl'
pool, meta = load_anchor_pool(POOL)
targets = [json.loads(l) for l in (ROOT / 'data/processed/requirements_v0.3.0.jsonl').read_text(encoding='utf-8').splitlines() if l.strip()]

print(f"검색기 설정: {retriever_config()}")
print(f"풀 {meta['reviewed_count']}건 / {meta['document_count']}문서 / sha {meta['sha256'][:16]}")
print(f"타깃 {len(targets):,}건")

In [ ]:
df_pool = pd.DataFrame(pool)

composition = pd.concat(
    [
        df_pool['primary_action'].value_counts().rename('건수').to_frame().assign(축='라벨'),
        df_pool['provenance'].value_counts().rename('건수').to_frame().assign(축='출처 등급'),
    ]
)
composition['비율_pct'] = (composition['건수'] / len(df_pool) * 100).round(1)
display(composition.set_index('축', append=True).swaplevel())

doc_label = pd.crosstab(df_pool['document_id'], df_pool['primary_action'])
doc_label['계'] = doc_label.sum(axis=1)
display(doc_label.sort_values('계', ascending=False))

shares = doc_label['계'] / doc_label['계'].sum()
print(f"문서 편중: 최대 점유율 {shares.max():.1%}, 문서당 {doc_label['계'].min()}~{doc_label['계'].max()}건")
print(f"데이터셋 대비 풀 비율: {len(pool) / len(targets):.1%}")
print('사람이 실제로 검토한 것은 사람확정 등급뿐입니다. 나머지는 반복 실행 일치로 걸러진 실버 라벨입니다.')

In [ ]:
retriever = PureTfidfAnchorRetriever(pool)

records = []
for target in targets:
    for rank, anchor in enumerate(retriever.retrieve(target, strategy='stratified'), 1):
        records.append({
            'target_uid': target['requirement_uid'],
            'target_doc': target['document_id'],
            'anchor_uid': anchor['requirement_uid'],
            'anchor_label': anchor['primary_action'],
            'similarity': anchor['similarity'],
        })
df = pd.DataFrame(records)

per_target = df.groupby('target_uid').size().value_counts().rename('타깃 수')
print('타깃별 주입 앵커 수:', per_target.to_dict())
print('층화가 성립하려면 모든 타깃이 3건이어야 합니다.\n')

usage = df['anchor_uid'].value_counts()
print(f"사용된 앵커 {usage.size}/{len(pool)}건, 미사용 {len(pool) - usage.size}건")
print(f"사용 횟수 중앙값 {usage.median():.0f} / 최대 {usage.max()} ({usage.max() / len(targets):.0%}의 타깃에 등장)\n")

print('인출 유사도')
display(df['similarity'].describe(percentiles=[0.25, 0.5, 0.75, 0.9]).round(3).to_frame().T)
for threshold in (0.05, 0.10, 0.20, 0.30):
    print(f"  유사도 < {threshold:.2f}: {(df['similarity'] < threshold).mean():.1%}")

top = usage.head(10).rename('사용 횟수').to_frame()
top = top.join(df_pool.set_index('requirement_uid')[['primary_action', 'requirement_name', 'document_id']])
display(top)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(19, 5))

doc_label.drop(columns='계').plot.barh(stacked=True, ax=axes[0], colormap='Set2')
axes[0].set(title='문서별 앵커 구성 (라벨 누적)', xlabel='건수', ylabel='')

axes[1].hist(df['similarity'], bins=60, color='#4C78A8')
axes[1].axvline(df['similarity'].median(), color='crimson', linestyle='--',
                label=f"중앙값 {df['similarity'].median():.3f}")
axes[1].set(title='인출 앵커 유사도 분포', xlabel='코사인 유사도', ylabel='인출 건수')
axes[1].legend()

for label, group in df.groupby('anchor_label'):
    axes[2].hist(group['similarity'], bins=40, alpha=0.55, label=label)
axes[2].set(title='라벨별 인출 유사도', xlabel='코사인 유사도', ylabel='인출 건수')
axes[2].legend()

plt.tight_layout()

print('층화 인출은 라벨마다 1건을 강제로 채웁니다.')
print('해당 라벨에 유사한 앵커가 없으면 가장 덜 무관한 것이 실리므로, 낮은 유사도 구간이 두꺼워집니다.')
print('이 분포가 앵커 풀 확장 여부를 판단하는 근거입니다.')